# PIKAN prediction for the semi-infinite-domain problem

In [1]:
import numpy as np

In [ ]:
from pathlib import Path
import sys
from importlib import reload

import matplotlib.pyplot as plt
import numpy as np
import torch

notebook_dir = Path.cwd().resolve()
repo_root = next(
    (path for path in [notebook_dir, *notebook_dir.parents] if (path / "utils").is_dir()),
    notebook_dir,
)
utilities_dir = repo_root / "utils"
if str(utilities_dir) not in sys.path:
    sys.path.insert(0, str(utilities_dir))

import pinns_infinite
import pinns_semi_infinite
import semi_infinite
reload(semi_infinite)
reload(pinns_infinite)
reload(pinns_semi_infinite)

from pinns_semi_infinite import build_models_KAN, set_seed, train_dual_network_semi_inf
from semi_infinite import (
    analytical_solution_semi_inf,
    coefficient_semi_inf,
    evaluate_model_semi_inf,
)

set_seed(42)
torch.set_default_dtype(torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

/home/orincon/miniconda3/envs/PIKAN-unbounded-domains-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## Tuned PIKAN configuration

In [ ]:
config = {
    # Choose "train" to fit a new model or "load" to use a stored checkpoint.
    "mode": "load",
    "checkpoint_name": "pikan_semi_infinite_tuned_weights.pt",
    "hidden_layers": 3,
    "hidden_units": 25,
    "grid_size": 5,
    "spline_order": 3,
    "adam_lr": 1e-3,
    "adam_iters": 2000,
    "lbfgs_iters": 2000,
    "sampling": "gaussian_exponential",
    "sigma": 5.5,
    "exp_scale": 7.0,
    "n_obs_u": 100,
    "n_boundary_u": 10,
    "n_obs_k": 100,
    "n_pde": 1000,
    "seed": 2,
    "pde_alpha": 0.5,
    "pde_beta": 5.0,
    "epsilon": 1.0,
    "train_domain": (-5.0, 5.0, -5.0, 0.0),
    "eval_domain": (-10.0, 10.0, -10.0, 0.0),
}

for name, value in config.items():
    print(f"{name}: {value}")

hidden_layers: 3
hidden_units: 25
grid_size: 5
spline_order: 3
adam_lr: 0.001
adam_iters: 2000
lbfgs_iters: 2000
sampling: gaussian_exponential
sigma: 5.5
exp_scale: 7.0
n_obs_u: 100
n_boundary_u: 10
n_obs_k: 100
n_pde: 1000
seed: 2
pde_alpha: 0.5
pde_beta: 5.0
epsilon: 1.0
train_domain: (-5.0, 5.0, -5.0, 0.0)
eval_domain: (-8.0, 8.0, -8.0, 0.0)


## Build the KAN models

In [3]:
model_u, model_k = build_models_KAN(
    device=device,
    hidden_layers=config["hidden_layers"],
    hidden_units=config["hidden_units"],
    grid_size=config["grid_size"],
    spline_order=config["spline_order"],
)

print(model_u)
print(model_k)

KAN(
  (layers): ModuleList(
    (0-3): 4 x KANLinear(
      (base_activation): SiLU()
    )
  )
)
KAN(
  (layers): ModuleList(
    (0-3): 4 x KANLinear(
      (base_activation): SiLU()
    )
  )
)


## Load the Gaussian semi-infinite PIKAN

In [ ]:
results_dir = repo_root / "main" / "03_individual_prediction" / "results"
weights_path = results_dir / config["checkpoint_name"]
available_checkpoints = sorted(results_dir.glob("*.pt"))
print("Available stored models:")
for checkpoint in available_checkpoints:
    print(f"  - {checkpoint.name}")

mode = config["mode"].lower()
if mode not in {"train", "load"}:
    raise ValueError('config["mode"] must be either "train" or "load"')

if mode == "load":
    if not weights_path.exists():
        raise FileNotFoundError(
            f"Stored checkpoint not found: {weights_path}. "
            f"Choose one of: {[path.name for path in available_checkpoints]}"
        )
    checkpoint = torch.load(weights_path, map_location=device)
    model_u.load_state_dict(checkpoint["model_u"])
    model_k.load_state_dict(checkpoint["model_k"])
    config.update(checkpoint.get("config", {}))
    metrics = checkpoint.get("metrics", {})
    print(f"Loaded stored model: {weights_path}")
else:
    history = train_dual_network_semi_inf(
        model_u,
        model_k,
        adam_lr=config["adam_lr"],
        adam_iters=config["adam_iters"],
        lbfgs_iters=config["lbfgs_iters"],
        verbose=True,
        print_every=100,
        save_every=100,
        lambda_pde_scheduler=True,
        adaptive_weights=True,
        alpha=7,
        update_every=100,
        regularization=False,
        sampling=config["sampling"],
        sigma=config["sigma"],
        exp_scale=config["exp_scale"],
        n_obs_u=config["n_obs_u"],
        n_boundary_u=config["n_boundary_u"],
        n_obs_k=config["n_obs_k"],
        n_pde=config["n_pde"],
        seed=config["seed"],
        save_results=True,
        base_dir=str(results_dir),
        run_name="pikan_semi_infinite_gaussian",
        pde_alpha=config["pde_alpha"],
        pde_beta=config["pde_beta"],
        epsilon=config["epsilon"],
        device=device,
    )

model_u.eval()
model_k.eval()
print(model_u)
print(model_k)

Loaded stored Gaussian semi-infinite checkpoint: /home/orincon/unbounded-domains/main/03_individual_prediction/results/pikan_semi_infinite_tuned_weights.pt


## Evaluate against the analytical solution

In [5]:
evaluation = evaluate_model_semi_inf(
    model_u=model_u,
    model_k=model_k,
    analytical_solution=analytical_solution_semi_inf,
    coefficient=coefficient_semi_inf,
    train_domain=config["train_domain"],
    eval_domain=config["eval_domain"],
    n_grid=400,
    alpha=config["pde_alpha"],
    beta=config["pde_beta"],
    epsilon=config["epsilon"],
    device=device,
    verbose=True,
)

metric_names = [
    "err_u_global", "err_k_global",
    "err_u_inside", "err_k_inside",
    "err_u_outside", "err_k_outside",
]
metrics = {name: float(evaluation[name]) for name in metric_names}
metrics

Semi-infinite-domain spatial generalization (MAE)
u (global): 4.326e-03
k (global): 9.246e-04
u (inside): 4.342e-03
k (inside): 8.306e-04
u (outside): 4.315e-03
k (outside): 9.848e-04


{'err_u_global': 0.004325610333373444,
 'err_k_global': 0.0009245634058072429,
 'err_u_inside': 0.004341629685017989,
 'err_k_inside': 0.0008306230731983915,
 'err_u_outside': 0.004315341518216685,
 'err_k_outside': 0.000984781567735994}